# 03 · WoE / IV binning
**Method.** From-scratch binner (`src/features/woe.py`): 20 quantile fine bins, then adjacent bins are merged until (a) every bin has >=5% of train and (b) bad-rate is **monotone**. NaN gets its own bin. `WoE = ln(%bad/%good)` with +0.5 smoothing, so a correct logistic model has **positive** coefficients.

### Why this way
- **Monotone coarse classing** makes each coefficient business-explainable and stops the champion from learning noise; a model reviewer can read a score card.
- **Own implementation + cross-check**: `scorecardpy` has had pandas/numpy compatibility issues, so the production path does not depend on it; it is used here only as an optional cross-check of IV.
- **Binning fitted on train only.** Validation is only ever *transformed*. The stability table below applies train bins to 2015 and checks that the risk ordering holds out-of-time.
- **IV thresholds:** drop `IV < 0.02` (no signal); flag `IV > 0.5` (suspiciously strong; investigate leakage). IV is a screening heuristic, not a significance test.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils.config import load_config, set_seeds
cfg = load_config(); set_seeds(cfg["seed"])
pd.set_option("display.float_format", "{:.4f}".format); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("config seed:", cfg["seed"], "| raw file:", cfg["paths"]["raw_csv"])

In [ ]:
from src.pipeline import prepare
from src.features.woe import WoEBinner, bin_stability
from src.features.guards import feature_sets
data = prepare(cfg)
num, cat = feature_sets(cfg)["with_lc"]
y = data.train.target.astype(int)
b = WoEBinner(cfg["binning"]["fine_bins"], cfg["binning"]["min_bin_frac"], num, cat).fit(data.train, y)
iv = b.iv_.to_frame("IV")
iv["decision"] = np.where(iv.IV < cfg["binning"]["iv_min"], "drop (IV too low)", np.where(iv.IV > cfg["binning"]["iv_suspicious"], "FLAG: too strong", "keep"))
iv["circular_LC"] = iv.index.isin(cfg["features"]["lc_numeric"] + cfg["features"]["lc_categorical"])
iv

Bin tables for key variables (train).

In [ ]:
for f in ["fico_mid","dti","revol_util"]:
    print(f); display(b.bin_table(f)[["bin","n","bad_rate","woe","iv"]])

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, f in zip(axs, ["fico_mid","dti","int_rate"]):
    t = b.bin_table(f); ax.bar(range(len(t)), t.woe, color=np.where(t.woe>0,"#e45756","#4c78a8")); ax.set_xticks(range(len(t))); ax.set_xticklabels(t.bin, rotation=60, ha="right", fontsize=7); ax.set_title(f"WoE: {f}")
plt.tight_layout(); plt.show()

**Binning stability out-of-time.** Train bins applied to 2015: does bad-rate keep the same ordering?

In [ ]:
yv = data.valid.target.astype(int)
for f in ["fico_mid","dti","revol_util"]:
    print(f); display(bin_stability(b, f, data.valid, yv))

**Optional cross-check against `scorecardpy`** (skipped automatically if not installed or incompatible).

In [ ]:
try:
    import scorecardpy as sc
    tmp = data.train[["fico_mid","dti","target"]].dropna().assign(target=lambda d: d.target.astype(int))
    bins = sc.woebin(tmp, y="target", positive="bad|1", print_info=False)
    ref = {k: v["total_iv"].iloc[0] for k, v in bins.items()}
    print(pd.DataFrame({"scorecardpy_IV": ref, "this_repo_IV": {k: b.iv_[k] for k in ref}}))
    print("IVs differ because the binning algorithms differ; the ranking of variables is what should agree.")
except Exception as e:
    print("scorecardpy cross-check skipped:", type(e).__name__, str(e)[:120])